# 03 — Explainable AI Analysis: SHAP and LIME

This notebook is the core contribution of the extension paper.

It applies SHAP and LIME to the 5 CSA-optimized classifiers trained in notebook 02.

**Key analyses:**
1. Global feature importance (SHAP bar + beeswarm for AB-CSA)
2. Per-class feature importance (which features detect which attack type?)
3. Cross-classifier comparison (do all 5 models agree on what matters?)
4. Failure analysis: why RF-CSA scores 0.00 F1 on Class 3 (Overflow)
5. LIME local explanations for selected samples
6. SHAP vs LIME consistency check

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib

from src.xai_analysis import (
    compute_shap_tree, compute_shap_linear, compute_shap_kernel,
    plot_shap_global, plot_shap_per_class, get_top_features_per_class,
    explain_with_lime, CLASS_NAMES
)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 150

In [ ]:
# Load trained models and preprocessed data from notebook 02
data = joblib.load("../results/preprocessed_data.pkl")
X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]
feature_names = data["feature_names"]

models = {}
for name in ["KNN", "LR", "RF", "DT", "AB"]:
    models[name] = joblib.load(f"../results/{name}_CSA_model.pkl")
    print(f"Loaded {name}-CSA model")

## 1. SHAP Global Analysis — AB-CSA (Best Model)

In [ ]:
# TreeExplainer for AdaBoost (best model at 88% accuracy)
explainer_ab, shap_values_ab = compute_shap_tree(models["AB"], X_test, "AB_CSA")
plot_shap_global(shap_values_ab, X_test, "AB_CSA")

## 2. SHAP for DT-CSA and RF-CSA

In [ ]:
explainer_dt, shap_values_dt = compute_shap_tree(models["DT"], X_test, "DT_CSA")
plot_shap_global(shap_values_dt, X_test, "DT_CSA")

explainer_rf, shap_values_rf = compute_shap_tree(models["RF"], X_test, "RF_CSA")
plot_shap_global(shap_values_rf, X_test, "RF_CSA")

## 3. SHAP for LR-CSA and KNN-CSA

In [ ]:
# LinearExplainer for Logistic Regression
explainer_lr, shap_values_lr = compute_shap_linear(models["LR"], X_train, X_test, "LR_CSA")
plot_shap_global(shap_values_lr, X_test, "LR_CSA")

In [ ]:
# KernelExplainer for KNN — slow, uses sampling
# Reduce n_explain if this takes too long (start with 100, increase if time permits)
explainer_knn, shap_values_knn = compute_shap_kernel(
    models["KNN"], X_train, X_test, "KNN_CSA",
    n_background=100, n_explain=200
)
plot_shap_global(shap_values_knn, X_test[:200], "KNN_CSA")

## 4. Per-Class Feature Importance (AB-CSA)

In [ ]:
plot_shap_per_class(shap_values_ab, X_test, "AB_CSA")

# Top-5 features per attack type (this becomes TABLE VI in the paper)
top_features_df = get_top_features_per_class(shap_values_ab, feature_names, top_n=5)
print(top_features_df.to_string(index=False))
top_features_df.to_csv("../results/top5_features_per_class_AB_CSA.csv", index=False)

## 5. Cross-Classifier SHAP Comparison Heatmap

In [ ]:
# Compare mean |SHAP| across classifiers
# For KNN, use only the subset that was explained
all_shap = {
    "AB-CSA": shap_values_ab,
    "DT-CSA": shap_values_dt,
    "RF-CSA": shap_values_rf,
}

importance_matrix = pd.DataFrame(index=feature_names)
for model_name, sv in all_shap.items():
    # Average across all classes, then mean absolute
    avg = np.mean([np.abs(sv[c]).mean(axis=0) for c in range(len(sv))], axis=0)
    importance_matrix[model_name] = avg

# Normalize per column for comparison
importance_norm = importance_matrix.div(importance_matrix.max(axis=0), axis=1)

fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(importance_norm.sort_values("AB-CSA", ascending=True),
            cmap="YlOrRd", annot=True, fmt=".2f", ax=ax)
ax.set_title("Normalized Feature Importance Across Classifiers")
ax.set_xlabel("Classifier")
plt.tight_layout()
plt.savefig("../figures/cross_classifier_shap_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Failure Analysis: RF-CSA vs AB-CSA on Class 3 (Overflow)

RF-CSA scored 0.00 precision/recall/F1 on Class 3 in the original paper.
AB-CSA scored 0.73/0.80/0.77. Why does RF fail where AB succeeds?

Compare SHAP values for Class 3 samples between the two models.

In [ ]:
# Filter to Class 3 (Overflow) samples in the test set
class3_mask = y_test == 3
class3_indices = np.where(class3_mask)[0]
print(f"Class 3 (Overflow) samples in test set: {class3_mask.sum()}")

# SHAP values for Class 3 prediction from RF vs AB
# shap_values[class_index] gives SHAP values for that output class
rf_class3_shap = shap_values_rf[3][class3_indices]  # RF's SHAP for predicting class 3
ab_class3_shap = shap_values_ab[3][class3_indices]  # AB's SHAP for predicting class 3

# Mean absolute SHAP for class 3 prediction
rf_importance = np.abs(rf_class3_shap).mean(axis=0)
ab_importance = np.abs(ab_class3_shap).mean(axis=0)

comparison_df = pd.DataFrame({
    "Feature": feature_names,
    "RF-CSA Mean|SHAP|": rf_importance,
    "AB-CSA Mean|SHAP|": ab_importance,
    "Difference (AB - RF)": ab_importance - rf_importance,
}).sort_values("Difference (AB - RF)", ascending=False)

print("\nFeatures where AB-CSA uses MORE signal than RF-CSA for Overflow detection:")
print(comparison_df.head(10).to_string(index=False))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
top_n = 10
top_feats = comparison_df.head(top_n)

axes[0].barh(range(top_n), top_feats["RF-CSA Mean|SHAP|"].values, color="salmon")
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels(top_feats["Feature"].values)
axes[0].set_title("RF-CSA: SHAP for Class 3 (F1=0.00)")
axes[0].set_xlabel("Mean |SHAP value|")

axes[1].barh(range(top_n), top_feats["AB-CSA Mean|SHAP|"].values, color="steelblue")
axes[1].set_yticks(range(top_n))
axes[1].set_yticklabels(top_feats["Feature"].values)
axes[1].set_title("AB-CSA: SHAP for Class 3 (F1=0.77)")
axes[1].set_xlabel("Mean |SHAP value|")

plt.suptitle("Why RF-CSA Fails on Overflow: Feature Contribution Comparison", fontsize=13)
plt.tight_layout()
plt.savefig("../figures/failure_analysis_class3.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. LIME Local Explanations

In [ ]:
# Select interesting samples for LIME
y_pred_ab = models["AB"].predict(X_test)
y_pred_rf = models["RF"].predict(X_test)

# Correctly classified attack sample (AB-CSA)
correct_attack = np.where((y_test == 1) & (y_pred_ab == 1))[0]
if len(correct_attack) > 0:
    exp1 = explain_with_lime(models["AB"], X_train, X_test,
                             correct_attack[0], feature_names, "AB_CSA_correct")

# Misclassified Overflow sample (RF-CSA failure case)
misclass_overflow = np.where((y_test == 3) & (y_pred_rf != 3))[0]
if len(misclass_overflow) > 0:
    exp2 = explain_with_lime(models["RF"], X_train, X_test,
                             misclass_overflow[0], feature_names, "RF_CSA_misclass_overflow")

# Same Overflow sample explained by AB-CSA (which gets it right)
if len(misclass_overflow) > 0:
    exp3 = explain_with_lime(models["AB"], X_train, X_test,
                             misclass_overflow[0], feature_names, "AB_CSA_same_overflow")

## 8. SHAP Dependence Plots (Top Features)

In [ ]:
# Dependence plots for the top-3 globally important features (AB-CSA)
global_importance = np.mean([np.abs(shap_values_ab[c]).mean(axis=0)
                             for c in range(len(shap_values_ab))], axis=0)
top3 = np.argsort(global_importance)[-3:][::-1]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, feat_idx in enumerate(top3):
    # Use class 0 SHAP values for dependence plot
    shap.dependence_plot(feat_idx, shap_values_ab[0], X_test,
                         feature_names=feature_names, ax=axes[i], show=False)
plt.suptitle("SHAP Dependence Plots — Top-3 Features (AB-CSA)", fontsize=13)
plt.tight_layout()
plt.savefig("../figures/shap_dependence_top3.png", dpi=300, bbox_inches="tight")
plt.show()

## 9. Summary — Export All Results

In [ ]:
print("\n=== XAI Analysis Complete ===")
print(f"Figures saved to: ../figures/")
print(f"SHAP values saved to: ../results/")
print(f"\nFigures generated:")
import os
for f in sorted(os.listdir("../figures/")):
    if f.endswith((".png", ".pdf")):
        print(f"  {f}")